In [ ]:
# first upload the diffpred_MODDED folder to the drive
# you need to have the test set for the code to work
# the dependencies are installed with curl and uv

In [ ]:
!curl -LsSf https://astral.sh/uv/0.10.5/install.sh | sh

downloading uv 0.10.5 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [ ]:
%cd /content/bac_thesis/diffpred_MODDED//diffpred

/content/bac_thesis/diffpred_MODDED/diffpred


In [ ]:
!uv sync

Using CPython 3.11.14
Creating virtual environment at: /content/bac_thesis/diffpred_MODDED/.venv
Resolved 363 packages in 1ms
Prepared 358 packages in 1m 29s
Installed 358 packages in 706ms
 + absl-py==2.4.0
 + accelerate==0.33.0
 + aimsim-core==2.2.3
 + aiobotocore==2.17.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aioitertools==0.13.0
 + aiosignal==1.4.0
 + alembic==1.18.4
 + anndata==0.11.4
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.12.1
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==25.1.0
 + array-api-compat==1.14.0
 + arrow==1.4.0
 + asciitree==0.3.3
 + astartes==1.3.3
 + asttokens==3.0.1
 + astunparse==1.6.3
 + async-lru==2.2.0
 + attrs==25.4.0
 + authlib==1.6.9
 + babel==2.18.0
 + backports-tarfile==1.2.0
 + beautifulsoup4==4.14.3
 + biotite==1.6.0
 + biotraj==1.2.2
 + bleach==6.3.0
 + boto3==1.35.93
 + botocore==1.35.93
 + botorch==0.16.1
 + cattrs==26.1.0
 + cellxgene-census==1.15.0
 + certifi==2026.2.25
 + cffi==2.0.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [ ]:
import wandb
wandb_key = userdata.get('WAND_API_KEY')
wandb.login(key=wandb_key)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: arri (arri-test) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# JKU TOX21 Experiments

# Hyperparameter Tuning

In [ ]:
!uv run python -u tune_tox21.py \
  --device cuda --data-dir data/tox21_jku \
  --output-root /content/drive/MyDrive/AI_thesis/diffpred_runs/tuning_unrounded

Output hidden; open in https://colab.research.google.com to view.

# Runs with multiple seeds

In [ ]:
from pathlib import Path
import subprocess

project = Path("/content/bac_thesis/diffpred_MODDED/diffpred")
runs = Path("/content/drive/MyDrive/AI_thesis/diffpred_runs")

sweep = runs / "outputs_tune/tox21/diffpred_manual_20261004_112746_832167"
output = runs / "depth4_5seeds"

seeds = [0, 42, 25, 17, 9]

models = {
    "chemprop": "best_params.yaml",
    "chemprop_diff": "best_params_diff.yaml",
}

for model, config in models.items():
    for seed in seeds:
        print(f"\nRunning {model}, seed {seed}", flush=True)

        subprocess.run(
            [
                "uv", "run", "python", "-u", "run.py",
                "--config-path", str(sweep),
                "--config-name", config,
                f"experiment.seed={seed}",
                "experiment.device=cuda",
                "experiment.wandb_mode=online",
                f"experiment.output_dir={output / model / f'seed_{seed}'}",
            ],
            cwd=project,
            check=True,
        )


Running chemprop, seed 0

Running chemprop, seed 42

Running chemprop, seed 25

Running chemprop, seed 17

Running chemprop, seed 9

Running chemprop_diff, seed 0

Running chemprop_diff, seed 42

Running chemprop_diff, seed 25

Running chemprop_diff, seed 17

Running chemprop_diff, seed 9


In [ ]:
###unrounded seeds###
from pathlib import Path
import subprocess

project = Path("/content/bac_thesis/diffpred_MODDED/diffpred")
runs = Path("/content/drive/MyDrive/AI_thesis/diffpred_runs")
sweep = runs / "outputs_tune/tox21/diffpred_manual_20261004_112746_832167"
output = runs / "depth4_5seeds_unrounded"

seeds = [0, 42, 25, 17, 9]

for seed in seeds:
    print(f"\nRunning unrounded Chemprop baseline, seed {seed}", flush=True)
    subprocess.run(
        [
            "uv", "run", "python", "-u", "run.py",
            "--config-path", str(sweep),
            "--config-name", "best_params.yaml",
            f"experiment.seed={seed}",
            "experiment.device=cuda",
            "experiment.wandb_mode=online",
            f"experiment.exp_name=jku_chemprop_unrounded_seed{seed}",
            f"experiment.output_dir={output / 'chemprop' / f'seed_{seed}'}",
        ],
        cwd=project,
        check=True,
    )


Running chemprop, seed 0

Running chemprop, seed 42

Running chemprop, seed 25

Running chemprop, seed 17

Running chemprop, seed 9

Running chemprop_diff, seed 0

Running chemprop_diff, seed 42

Running chemprop_diff, seed 25

Running chemprop_diff, seed 17

Running chemprop_diff, seed 9


# Sigmoid Rounding

In [ ]:
!uv run python -u logit_check.py \
  --runs-dir /content/drive/MyDrive/AI_thesis/diffpred_runs/depth4_5seeds/chemprop \
  --data-dir data/tox21_jku \
  --device cuda

Logit-rounding check: 5 baseline checkpoint(s)
Output: /content/drive/MyDrive/AI_thesis/diffpred_runs/logit_check_run/check_20261009_103344_360809_UTC
Fixed-checkpoint sensitivity analysis: both score variants use identical raw logits, records, and observed-label masks. The original hyperparameters and checkpoints were selected using rounded validation scores. No retraining, new checkpoint selection, split changes, or test-record imputation is performed.
I0000 00:00:1791542038.757893    7693 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791542038.835381    7693 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, r